In [1]:
import pandas as pd
from sklearn.datasets import load_iris
import sqlalchemy
from sqlalchemy import create_engine, text, MetaData, Table

In [2]:
# Environment variables
import os
import dotenv

dotenv.load_dotenv()  
USER = os.getenv('USER')
PASSWORD = os.getenv('PASSWORD')
HOST = os.getenv('HOST')

In [3]:
# load data
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df.columns = ['sepal_length','sepal_width', 'petal_length', 'petal_width']
df['target'] = iris.target
df.head()

,sepal_length,sepal_width,petal_length,petal_width,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [4]:
df2 = df.copy()

In [5]:
def create_db(engine: sqlalchemy.engine.Engine, database_name: str) -> None:
    """
    Creates the specified database if it doesn't exist.

    Args:
        engine (sqlalchemy.engine.Engine): The SQLAlchemy engine object.
        database_name: The name of the database to create.

    """

    with engine.connect() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {database_name}"))
        print(f"Database '{database_name}' created or already exists.")

def drop_db(engine: sqlalchemy.engine.Engine, database_name: str) -> None:
    """
    Drops the specified database if it exists.

    Args:
        engine (sqlalchemy.engine.Engine): The SQLAlchemy engine object.
        database_name: The name of the database to drop.

    """

    with engine.connect() as conn:
        conn.execute(text(f"DROP DATABASE {database_name}"))
        conn.commit()
        print(f"Database '{database_name}' dropped or did not exist.")

In [6]:
# Crear conexión con el servidor (sin especificar la base de datos)
engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}')

create_db(engine, 'stories')

Database 'stories' created or already exists.


In [7]:
def list_tables(engine: sqlalchemy.engine.Engine):
    """
    Gets a list of all tables in the database.

    Args:
        engine (sqlalchemy.engine.Engine): The SQLAlchemy engine object.

    Returns:
        list[str]: A list of table names.
    """

    with engine.connect() as conn:
        result = conn.execute(text("SHOW TABLES"))
        table_names = [row[0] for row in result]

    print(f"MySQL tables:")
    return table_names

def drop_table(engine: sqlalchemy.engine.Engine, 
               table_name: str) -> None:
    """
    Drops the specified table from the database.

    Args:
        engine (sqlalchemy.engine.Engine): The SQLAlchemy engine object.
        table_name (str): The name of the MySQL table to drop.

    Raises:
        sqlalchemy.exc.OperationalError: If an error occurs during the table drop operation.

    Returns:
        None
    """
    try:
        metadata = MetaData()
        table = Table(table_name, metadata, autoload_with=engine)
        table.drop(engine)
        print(f"Table '{table_name}' dropped successfully.")
        print(list_tables(engine))
    except sqlalchemy.exc.OperationalError as e:
        print(f"Error dropping table '{table_name}': {e}")

def upload_data_mysql(engine: sqlalchemy.engine.Engine, 
                      dataframe: pd.DataFrame,
                      table_name: str, 
                      chunksize: int = None
                      ) -> None:

    """
    Uploads a Pandas DataFrame to a MySQL table, handling potential errors and providing informative feedback.

    Args:
        engine (sqlalchemy.engine.Engine): The SQLAlchemy engine object.
        dataframe (pd.DataFrame): The DataFrame to upload.
        table_name (str, optional): The name of the MySQL table.
        chunksize (int, optional): The size of the data chunks to be uploaded. Defaults to 10000.

    Returns:
        None
    """

    try:
        dataframe.to_sql(table_name, con=engine, if_exists='append', index=False, chunksize=chunksize)
        print(f"Data uploaded successfully to MySQL table '{table_name}'.")
    except Exception as e:
        print(f"Error uploading data to MySQL: {e}")
        raise  # Re-raise the exception for further handling 

def read_table(engine: sqlalchemy.engine.Engine, table_name: str) -> pd.DataFrame:
    """
    Reads the specified table from the database.

    Args:
        engine (create_engine.Engine): The SQLAlchemy engine object.
        table_name (str): The name of the table to load.

    Returns:
        pd.DataFrame: The loaded table as a pandas DataFrame.
    """

    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql_query(query, engine)
    print(f"Loaded table: {table_name}")
    return df

In [8]:
# Conectar a la base de datos 'stores'
engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}/stories')

In [9]:
list_tables(engine)

MySQL tables:


[]

In [10]:
upload_data_mysql(engine, df, 'hh')

Data uploaded successfully to MySQL table 'hh'.


In [11]:
upload_data_mysql(engine, df2, 'hh')

Data uploaded successfully to MySQL table 'hh'.


In [12]:
df_hh = read_table(engine, 'hh')

Loaded table: hh


In [13]:
df_hh.shape

(300, 5)

In [14]:
drop_db(engine, 'stories')

Database 'stories' dropped or did not exist.
